# Digital signatures  (MODEL SOLUTION, instructor copy, do not distribute)

# Coding problem, 5.4.c: Implement RSA digital signatures (hash-then-sign)

**Skill:** Mitigate Risk · **Maps to:** 5.4.B · **Language:** Python 3 (pytest)

## Problem
A software vendor wants to let users verify that a downloaded update really came from them and
was not modified in transit. Rather than signing the whole update file directly with RSA (slow,
and structurally risky per atom.md), implement the standard "hash, then sign" digital signature
scheme: hash the message with SHA-256, reduce the digest mod n, and apply the RSA private-key
operation to that digest, not the raw message.

## Function contract
Implement, in the code editor:
- `sign(message: bytes, d: int, n: int) -> int`, return the signature
  `(SHA-256(message) mod n)^d mod n`, using the signer's private exponent `d` and modulus `n`.
- `verify(message: bytes, signature: int, e: int, n: int) -> bool`, recompute
  `SHA-256(message) mod n` and compare it to `signature^e mod n`, using the signer's public
  exponent `e` and modulus `n`. Return `True` iff they match.
- `solve(data: dict) -> dict`, the blackbox entry point that dispatches to the two functions
  above (see the Reference section below).

You may use `hashlib.sha256` for the hash step only. Implement the digest-reduction and the
modular exponentiation (`pow(base, exp, mod)` is fine, it's Python's built-in, not an external
crypto library) yourself, per the construction below.

## The construction
```
digest        = int.from_bytes(SHA256(message).digest(), "big") % n
sign:         signature = digest^d mod n
verify:       recovered = signature^e mod n     ; valid iff recovered == digest (recomputed)
```
- Hashing first keeps the signature a fixed size regardless of message length, and means a
  single changed byte in `message` produces a completely different digest (avalanche effect,
  5.3.b), which is what makes tampering detectable.
- `d` and `n` form the private key; `e` and `n` form the matching public key (5.4.b).

## Input / Output
- `sign(message, d, n)`: `message: bytes`, `d: int`, `n: int` -> `int` signature.
- `verify(message, signature, e, n)`: `signature: int`, `e: int`, `n: int` -> `bool`.
- `solve(data)`: `data = {"op": "sign", "message": "<str>", "d": <int>, "n": <int>}` ->
  `{"signature": <int>}`, or
  `data = {"op": "verify", "message": "<str>", "signature": <int>, "e": <int>, "n": <int>}` ->
  `{"valid": <bool>}`.

## Examples
Using the textbook-scale key pair `p=61, q=53 -> n=3233`, `e=17`, `d=2753`:
```
sign(b"pay Bob $50", 2753, 3233)        -> some int in [0, 3233)
verify(b"pay Bob $50", <that signature>, 17, 3233)   -> True
verify(b"pay Bob $950", <that signature>, 17, 3233)  -> False   # message tampered after signing
```

## Constraints
- Deterministic, offline, standard library only (`hashlib` only, no external crypto packages).
- Correctness is checked in the tests against an independent oracle that recomputes the same
  hash-then-sign math directly (not by importing your functions).
- Must pass all cases in the tests (visible + hidden), including: a tampered message failing
  verification, a wrong public key failing verification, an empty message, and determinism
  (signing the same message twice yields the same signature).

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry points
```python
def sign(message: bytes, d: int, n: int) -> int
def verify(message: bytes, signature: int, e: int, n: int) -> bool
def solve(data: dict) -> dict
```

`solve()` is the blackbox entry point; `sign()` and `verify()` are the underlying primitives
it's built on and are tested directly as well.

### Input
- `sign(message, d, n)`: `message` is raw `bytes` of any length (including empty); `d` is the
  signer's private exponent; `n` is the RSA modulus (both small, textbook-scale ints).
- `verify(message, signature, e, n)`: `message` as above; `signature` is the claimed signature
  (`int`); `e` is the public exponent; `n` is the modulus that must match the key pair used to
  sign (verifying with a different `n`/`e` pair must fail).
- `solve(data)`: `data` is a `dict` with `"op"` equal to `"sign"` or `"verify"`.
  - `"sign"`: `{"op": "sign", "message": <str>, "d": <int>, "n": <int>}`
  - `"verify"`: `{"op": "verify", "message": <str>, "signature": <int>, "e": <int>, "n": <int>}`
  - `message` arrives as a plain UTF-8 string (encode with `.encode("utf-8")`).

### Output (`result`)
- `sign(...)` returns an `int` signature in the range `[0, n)`.
- `verify(...)` returns `bool`.
- `solve(...)` returns `{"signature": <int>}` for op `"sign"`, or `{"valid": <bool>}` for op
  `"verify"`.

### Guarantees / rules
- Pure functions, deterministic, no I/O, no network. Standard library only (`hashlib.sha256`
  may be used as the hash primitive; the RSA sign/verify math itself, reducing the digest mod
  `n` and using modular exponentiation, e.g. Python's built-in `pow(base, exp, mod)`, must be
  implemented directly, not via any external crypto library).
- Must implement "hash, then sign": `signature = (SHA-256(message) mod n)^d mod n`; verification
  recomputes `SHA-256(message) mod n` and compares it to `signature^e mod n`.
- Any single-bit change to `message` must (with overwhelming probability) produce a different
  signature, and verification against the original signature must then fail.
- Verifying a genuine `(message, signature)` pair with an unrelated key pair's public key
  `(e, n)` must return `False`.

### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).


In [ ]:
# Reference solution
"""Reference solution for 5.4.c. MUST pass 100% of the tests.
Also the default 'blackbox' implementation students can swap out (see the hidden grading tests).

Implements textbook RSA "hash-then-sign" digital signatures by hand: the digest is
computed with hashlib.sha256 (stdlib), reduced mod n, and the RSA private/public-key
operation (modular exponentiation) is implemented directly with Python's built-in
pow(base, exp, mod) rather than any external crypto library.
Crypto problems implement the underlying math/logic directly.
"""
import hashlib


def _digest_mod_n(message: bytes, n: int) -> int:
    """Hash message with SHA-256 and reduce it to an integer mod n.

    This is the 'hash, then sign' step from atom.md: we never run RSA on the raw
    message, only on a fixed-size digest of it.
    """
    digest_bytes = hashlib.sha256(message).digest()
    digest_int = int.from_bytes(digest_bytes, "big")
    return digest_int % n


def sign(message: bytes, d: int, n: int) -> int:
    """Sign `message` with private exponent d and modulus n.

    signature = hash(message)^d mod n
    """
    digest = _digest_mod_n(message, n)
    return pow(digest, d, n)


def verify(message: bytes, signature: int, e: int, n: int) -> bool:
    """Verify that `signature` is a valid signature over `message` under public
    key (e, n).

    Recomputes hash(message) mod n and compares it to signature^e mod n.
    """
    expected_digest = _digest_mod_n(message, n)
    recovered_digest = pow(signature, e, n)
    return expected_digest == recovered_digest


def solve(data: dict) -> dict:
    """Blackbox entry point. Dispatches on data['op']:
      - {"op": "sign", "message": <str>, "d": <int>, "n": <int>}
            -> {"signature": <int>}
      - {"op": "verify", "message": <str>, "signature": <int>, "e": <int>, "n": <int>}
            -> {"valid": <bool>}
    Messages are given as UTF-8 strings; d/e/n/signature are given as ints (small,
    textbook-scale RSA parameters — JSON-friendly, no hex/bignum encoding needed).
    """
    op = data["op"]
    message = data["message"].encode("utf-8")
    if op == "sign":
        signature = sign(message, data["d"], data["n"])
        return {"signature": signature}
    elif op == "verify":
        valid = verify(message, data["signature"], data["e"], data["n"])
        return {"valid": valid}
    else:
        raise ValueError(f"unknown op: {op!r}")


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

import hashlib


# Textbook-scale RSA key pair, independent of implementation under test:
# p = 61, q = 53 -> n = 3233, phi(n) = 3120; e = 17, d = 2753 (17*2753 mod 3120 == 1)
N = 3233
E = 17
D = 2753

def _oracle_digest_mod_n(message: bytes, n: int) -> int:
    """Independent oracle: hash(message) reduced mod n, computed directly here
    (not reusing impl's internals) so tests don't just check self-consistency."""
    digest_bytes = hashlib.sha256(message).digest()
    return int.from_bytes(digest_bytes, "big") % n


def _oracle_sign(message: bytes, d: int, n: int) -> int:
    return pow(_oracle_digest_mod_n(message, n), d, n)


def _find_d(e: int, phi_n: int) -> int:
    """Brute-force modular inverse for small phi_n, used only to build test fixtures."""
    for cand in range(2, phi_n):
        if (e * cand) % phi_n == 1:
            return cand
    raise ValueError("no modular inverse found")


# A second, unrelated RSA key pair used to simulate "wrong public key" verification.
_P2, _Q2 = 59, 61
N2 = _P2 * _Q2  # 3599
_PHI2 = (_P2 - 1) * (_Q2 - 1)  # 3480
E2 = 7
D2 = _find_d(E2, _PHI2)


# ---- visible cases -------------------------------------------------------
def test_sign_matches_oracle():
    message = b"transfer $100 to acct 42"
    got = impl.sign(message, D, N)
    assert got == _oracle_sign(message, D, N)


def test_valid_signature_verifies_true():
    message = b"pay Bob $50"
    signature = impl.sign(message, D, N)
    assert impl.verify(message, signature, E, N) is True


def test_tampered_message_fails_verification():
    message = b"pay Bob $50"
    tampered = b"pay Bob $950"
    signature = impl.sign(message, D, N)
    assert impl.verify(tampered, signature, E, N) is False


def test_wrong_public_key_fails_verification():
    message = b"deploy build 42"
    signature = impl.sign(message, D, N)
    # Verifying with an unrelated key pair's public key must fail.
    assert impl.verify(message, signature, E2, N2) is False


def test_solve_sign_op():
    result = impl.solve({"op": "sign", "message": "hello", "d": D, "n": N})
    expected = _oracle_sign(b"hello", D, N)
    assert result == {"signature": expected}


def test_solve_verify_op_valid():
    message = "audit log entry 7"
    signature = _oracle_sign(message.encode("utf-8"), D, N)
    result = impl.solve({"op": "verify", "message": message, "signature": signature, "e": E, "n": N})
    assert result == {"valid": True}


# ---- hidden / edge cases -------------------------------------------------
def test_solve_verify_op_tampered_message():
    message = "audit log entry 7"
    signature = _oracle_sign(message.encode("utf-8"), D, N)
    result = impl.solve(
        {"op": "verify", "message": "audit log entry 8", "signature": signature, "e": E, "n": N}
    )
    assert result == {"valid": False}


def test_single_bit_change_in_message_flips_signature():
    a = b"password reset for user 1000"
    b = b"password reset for user 1001"  # differs by a single character
    sig_a = impl.sign(a, D, N)
    sig_b = impl.sign(b, D, N)
    assert sig_a != sig_b


def test_empty_message_signs_and_verifies():
    message = b""
    signature = impl.sign(message, D, N)
    assert impl.verify(message, signature, E, N) is True


def test_verify_rejects_signature_from_different_message_same_key():
    msg1 = b"approve invoice 9001"
    msg2 = b"approve invoice 9002"
    sig1 = impl.sign(msg1, D, N)
    assert impl.verify(msg2, sig1, E, N) is False


def test_signature_is_deterministic():
    message = b"same input, same output"
    sig1 = impl.sign(message, D, N)
    sig2 = impl.sign(message, D, N)
    assert sig1 == sig2


def test_signature_in_valid_range():
    message = b"range check"
    signature = impl.sign(message, D, N)
    assert 0 <= signature < N


def test_solve_unknown_op_raises():
    import pytest

    with pytest.raises(Exception):
        impl.solve({"op": "bogus", "message": "x"})


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
